# Подготовка текстовых данных

В электронную приёмную приходят обращения о библиотеке, общежитии, поступлении и работе учебного портала. Позже мы обучим модель направлять их в нужный отдел. Сначала разберёмся с самими сообщениями: пустые строки, повторы и неверные метки могут повлиять на результат ещё до обучения.

В этой работе вы проверите небольшой набор обращений и подготовите функцию обработки текста. Исходные файлы сохраняйте: исправления будем записывать в новые столбцы.

In [1]:
from pathlib import Path
import os
import json
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Находим корень раздачи независимо от места запуска Jupyter.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data" / "requests.csv").exists())
DATA = ROOT / "data"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
np.random.seed(42)
plt.rcParams.update({"font.family": "DejaVu Sans", "figure.figsize": (8, 4)})


## Данные

В `requests.csv` собраны обращения с шестью тематическими метками. `group_id` объединяет разные формулировки одного обращения. В отдельном файле `audit_requests.csv` есть несколько ошибок — их и предстоит найти. Описание полей и список меток лежат в `manifest.json`.

Это учебные данные, составленные для курса. Посмотрите на первые строки и попробуйте заметить подозрительные записи до того, как начнёте писать проверки.

In [2]:
requests = pd.read_csv(DATA / "requests.csv", keep_default_na=False)
audit = pd.read_csv(DATA / "audit_requests.csv", keep_default_na=False)
manifest = json.loads((DATA / "manifest.json").read_text(encoding="utf-8"))
required = {"id", "text", "label", "group_id", "split"}
assert required <= set(requests.columns), "Неверная схема основного корпуса"
assert required <= set(audit.columns), "Неверная схема файла аудита"
print("Основной корпус:", requests.shape)
print("Аудит:", audit.shape)
display(audit.head())

Основной корпус: (600, 5)
Аудит: (24, 5)


,id,text,label,group_id,split
0,it-01-1,После смены пароля не могу войти в личный каби...,it,it-01,train
1,it-01-2,Здравствуйте! После смены пароля не могу войти...,it,it-01,train
2,it-01-3,"Подскажите, пожалуйста. После смены пароля не ...",it,it-01,train
3,library-01-1,Как продлить срок пользования учебником?,library,library-01,train
4,library-01-2,Здравствуйте! Как продлить срок пользования уч...,library,library-01,train


### Сначала небольшой пример

Пустой текст не всегда выглядит как пустая строка. Сравним длину трёх значений до и после `strip()`. Какое из них проверка по исходной длине пропустила бы?

In [3]:
sample = pd.Series(["", "   ", "Ошибка 403"], name="text")
display(pd.DataFrame({"text":sample.map(repr),
                      "length":sample.str.len(),
                      "length_after_strip":sample.str.strip().str.len()}))

,text,length,length_after_strip
0,'',0,0
1,' ',3,0
2,'Ошибка 403',10,10


У строки из трёх пробелов исходная длина равна 3, а после `strip()` — 0. Сообщение с кодом ошибки остаётся непустым. Используйте это различие при проверке полной таблицы.

## 1. Что находится в корпусе

Посчитайте строки и группы обращений в основной таблице. Сколько примеров относится к каждой категории? Сравните эти числа: пять перефразировок одного вопроса дают пять строк, но описывают одну ситуацию.

Теперь проверьте файл аудита. Найдите строки без текста и метки, которых нет в списке `manifest["labels"]`. Строку из одних пробелов также считайте пустой.

Запишите результаты в словарь `summary` с полями `rows`, `groups`, `labels`, `empty_rows` и `unknown_labels`. В `labels` поместите частоты категорий, в `empty_rows` — целое число пустых строк, в `unknown_labels` — список неизвестных меток. Выведите сами подозрительные записи, чтобы было понятно, что нашла проверка.

In [4]:
empty = audit["text"].str.strip().eq("")
unknown = ~audit["label"].isin(manifest["labels"])
summary = {
    "rows": len(requests),
    "groups": requests["group_id"].nunique(),
    "labels": requests["label"].value_counts().to_dict(),
    "empty_rows": int(empty.sum()),
    "unknown_labels": sorted(audit.loc[unknown, "label"].unique().tolist()),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
display(audit.loc[empty | unknown])

{
  "rows": 600,
  "groups": 120,
  "labels": {
    "it": 100,
    "library": 100,
    "dormitory": 100,
    "study": 100,
    "admission": 100,
    "scholarship": 100
  },
  "empty_rows": 2,
  "unknown_labels": [
    "unknown"
  ]
}


,id,text,label,group_id,split
19,audit-empty,,it,audit-b,audit
20,audit-spaces,\t,study,audit-c,audit
22,audit-unknown,Как войти в кабинет?,unknown,audit-e,audit


In [5]:
assert summary["rows"] == len(requests)
assert sum(summary["labels"].values()) == len(requests)
assert isinstance(summary["empty_rows"], int)

В основной таблице 600 строк и 120 групп; на каждую из шести категорий приходится 100 строк. В аудите найдутся две пустые записи и метка unknown. Здесь полезно попросить студента показать найденные строки: одно итоговое число не объясняет, что именно он посчитал.

## 2. Пробелы и Unicode

Начнём с обработки, которая не меняет содержание обращения. Например, строка `"  Не  отменяйте запись!  "` должна превратиться в `"Не отменяйте запись!"`: лишние пробелы исчезли, отрицание и знак остались.

Напишите `normalize_text(text)`. Функция должна приводить Unicode к NFC, заменять последовательность пробельных символов одним пробелом и убирать пробелы по краям. Если передано не строковое значение, вызовите `TypeError`. Регистр, цифры и пунктуацию сохраняйте.

Примените функцию к `audit["text"]`, сохранив результат в новом столбце `audit["normalized"]`. Сопоставьте исходные и обработанные строки: какие из них изменились?

In [6]:
def normalize_text(text):
    if not isinstance(text, str):
        raise TypeError("Ожидалась строка")
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", text)).strip()

original = audit["text"].copy()
audit["normalized"] = audit["text"].map(normalize_text)
display(audit.loc[audit["text"].ne(audit["normalized"]), ["id", "text", "normalized"]])

,id,text,normalized
18,audit-duplicate,После смены пароля не могу войти в личный каби...,После смены пароля не могу войти в личный каби...
20,audit-spaces,\t,


In [7]:
for value in ["  Не  отменяйте!  ", "Ошибка 403", "и\u0306", ""]:
    result = normalize_text(value)
    assert normalize_text(result) == result  # идемпотентность
assert normalize_text("  Не  отменяйте!  ") == "Не отменяйте!"
assert normalize_text("Ошибка 403") == "Ошибка 403"
assert normalize_text("и\u0306") == "й"
assert audit["text"].equals(original)
try:
    normalize_text(None)
except TypeError:
    pass
else:
    raise AssertionError("Нестрочный вход должен вызывать TypeError")

В NFC последовательность и + комбинируемый знак краткости превращается в й. Это другой случай, чем латинская a вместо кириллической а: похожие буквы из разных алфавитов NFC не заменяет. Повторная нормализация уже обработанной строки ничего не меняет.

## 3. Повторы и разметка

После нормализации некоторые записи могут оказаться одинаковыми. Найдите такие строки среди непустых текстов и сохраните их в `duplicate_rows`. Нужны все участники каждого повтора, включая первую запись.

Для каждого текста посчитайте число разных меток. Оставьте случаи, где их больше одной, в `conflicts`: индекс — нормализованный текст, значение — число меток.

Прочитайте найденные обращения. Где достаточно убрать копию, а где придётся выяснять правильную категорию? Посмотрите и на остальные записи аудита: совпадение текстов — не единственный способ заметить ошибку разметки.

In [8]:
nonempty = audit.loc[audit["normalized"].ne("")].copy()
duplicate_rows = nonempty.loc[nonempty.duplicated("normalized", keep=False)]
label_counts = nonempty.groupby("normalized")["label"].nunique()
conflicts = label_counts[label_counts > 1]
display(duplicate_rows[["id", "normalized", "label"]])
print("Конфликтные тексты:")
display(conflicts)

,id,normalized,label
0,it-01-1,После смены пароля не могу войти в личный каби...,it
18,audit-duplicate,После смены пароля не могу войти в личный каби...,it
23,audit-conflict,После смены пароля не могу войти в личный каби...,library


Конфликтные тексты:


normalized
После смены пароля не могу войти в личный кабинет.    2
Name: label, dtype: int64

In [9]:
assert not duplicate_rows["normalized"].eq("").any()
assert {"it-01-1", "audit-duplicate", "audit-conflict"} <= set(duplicate_rows["id"])
assert len(conflicts) > 0 and (conflicts > 1).all()

Три записи относятся к одному нормализованному тексту, которому присвоены две разные метки. Кроме этого, audit-label содержит вопрос о продлении книги с меткой общежития. В обсуждении стоит отделить технический поиск совпадений от решения о категории: последнее требует чтения обращения и правил разметки.

## 4. Где проходит граница токена

Сравним два способа разбить сообщение на части. `split()` делит строку по пробелам. Выражение `re.findall(r"\w+|[^\w\s]", text)` выделяет последовательности букв, цифр и подчёркиваний, а также отдельные знаки.

Для каждого примера создайте строку таблицы `token_report`: исходный `text`, токены `split` и `regex`, их количества `n_split` и `n_regex`, а также число разных токенов `vocab_regex`.

Обратите внимание на `help@sever.example`: остался ли адрес одним элементом? Какие элементы относятся к самому адресу, а какие — к окружающей пунктуации? Что нужно добавить к разбиению, чтобы получить адрес целиком и без запятой?

In [10]:
examples = ["ИТ-поддержка: вход не работает!",
            "Письмо: help@sever.example, ошибка 403",
            "книга книга журнал"]
rows = []
for text in examples:
    a = text.split()
    b = re.findall(r"\w+|[^\w\s]", text)
    rows.append({"text": text, "split": a, "regex": b,
                 "n_split": len(a), "n_regex": len(b), "vocab_regex": len(set(b))})
token_report = pd.DataFrame(rows)
display(token_report)

,text,split,regex,n_split,n_regex,vocab_regex
0,ИТ-поддержка: вход не работает!,"[ИТ-поддержка:, вход, не, работает!]","[ИТ, -, поддержка, :, вход, не, работает, !]",4,8,8
1,"Письмо: help@sever.example, ошибка 403","[Письмо:, help@sever.example,, ошибка, 403]","[Письмо, :, help, @, sever, ., example, ,, оши...",4,10,10
2,книга книга журнал,"[книга, книга, журнал]","[книга, книга, журнал]",3,3,2


In [11]:
assert len(token_report) == len(examples)
assert all(len(tokens) == n for tokens, n in zip(token_report["regex"], token_report["n_regex"]))

В первой строке split даёт 4 токена, регулярное выражение — 8. В строке «книга книга журнал» три вхождения, но только два разных токена. Адрес разбивается на несколько частей; для извлечения адресов понадобится правило, которое учитывает их структуру.

## 5. Что теряется при очистке текста

Проверим распространённую идею: оставить только слова, перевести их в нижний регистр и удалить частые служебные элементы.

Напишите `aggressive(text)`: выделите токены через `re.findall(r"\w+", text.lower())`, удалите «я», «не» и токены, состоящие только из цифр. Остальное соедините пробелами.

Сначала предскажите результат для трёх строк ниже, затем выполните код. Можно ли по полученным строкам отличить довольного пользователя от недовольного? Что теперь известно о технической ошибке?

In [12]:
def aggressive(text):
    tokens = re.findall(r"\w+", text.lower())
    return " ".join(t for t in tokens if t not in {"я", "не"} and not t.isdigit())

texts = ["Я доволен работой портала", "Я не доволен работой портала", "Ошибка 403"]
for text in texts:
    print(text, "=>", aggressive(text))

Я доволен работой портала => доволен работой портала
Я не доволен работой портала => доволен работой портала
Ошибка 403 => ошибка


In [13]:
assert aggressive(texts[0]) == aggressive(texts[1])
assert "403" not in aggressive(texts[2])

Противоположные отзывы превращаются в одну строку. «Ошибка 403» теряет номер. Студент должен объяснить потерю конкретного признака: отрицание определяет отношение пользователя, а номер помогает отличить одну ошибку от другой.

## Одной форме могут соответствовать разные слова

Попробуем связать словоформы с леммами. Для «книги» и «книгой» в нашем маленьком словаре один кандидат. Для «стали» — два. Прочитайте оба предложения под выводом и выберите подходящую лемму для каждого. Какие соседние слова помогли сделать выбор?

In [14]:
# Ограниченный учебный словарь: демонстрирует неоднозначность, не заменяет анализатор.
lemma_candidates = {"книги": ["книга"], "книгой": ["книга"], "стали": ["сталь", "стать"]}
for token, candidates in lemma_candidates.items():
    print(token, "=>", candidates)
print("Разберите по контексту: Образцы стали / Студенты стали проверять")

книги => ['книга']
книгой => ['книга']
стали => ['сталь', 'стать']
Разберите по контексту: Образцы стали / Студенты стали проверять


## Разбор результатов

В аудите обнаружены две пустые строки, неизвестная категория, повтор одного обращения и конфликт его меток. Ещё одну ошибку видно при чтении: вопрос о книге направлен в общежитие. Эти случаи требуют разных решений — заполнить или исключить пустую запись, проверить категорию, выяснить происхождение копий.

Минимальная нормализация убрала различия в пробелах и записи Unicode. Агрессивная обработка затронула смысл: «Я доволен…» и «Я не доволен…» стали неразличимы, а 403 исчезло. Это хороший момент вернуться к будущей задаче модели: какие свойства сообщения ей понадобятся для ответа?

В морфологическом примере «Образцы стали» относится к материалу, а «Студенты стали проверять» — к действию. Список кандидатов помогает увидеть неоднозначность; выбрать лемму позволяет контекст.

Для короткой защиты можно дать строку `  Не  отменяйте запись на 12 октября! `. Ожидаемая нормализация — `Не отменяйте запись на 12 октября!`. Попросите объяснить, почему сохранились отрицание, дата и пунктуация.